# Task 1: CCG vs BDCP Consistency Verification

This notebook verifies that the Column-and-Constraint Generation (C&CG) and
Benders Decomposition Cutting Plane (BDCP) algorithms produce consistent results
on the same problem instances.

Both algorithms solve the two-stage robust CFLP (model 12 in the paper) and should
converge to the same optimal profit within the specified tolerance.

**Consistency checks:**
1. Optimal objective values agree within 1% (the solver tolerance).
2. Both algorithms report convergence.
3. The first-stage solutions x_jr are identical or yield the same profit.
4. The worst-case recourse costs (upper bounds) agree.

**Reference:** Algorithm 1 (C&CG) and Section 2.3 (BDCP) of the paper.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from rcflp import instancemaker, solve_nominal, solve_CCG, solve_BDCP

## 1. Experiment configuration

We test on a grid of small instances to keep runtimes manageable while covering
different parameter settings (instance size, uncertainty budget, congestion cost,
willingness-to-pay level).

In [ ]:
# ── Instance parameters ────────────────────────────────────────────────────
CONFIGS = [
    # (In, Jn, Rn, value_max_scale, congestion_cost, Gamma, Hn, label)
    (5,  5, 2, 0.75, 10,  1, 2, 'small-Γ1'),
    (5,  5, 2, 0.75, 10,  2, 2, 'small-Γ2'),
    (10, 8, 2, 0.75, 10,  1, 2, 'medium-Γ1'),
    (10, 8, 2, 0.75, 10,  2, 2, 'medium-Γ2'),
    (10, 8, 2, 1.00, 100, 2, 2, 'medium-highW'),
    (10, 8, 3, 0.75, 10,  2, 3, 'medium-R3-H3'),
]

# ── Solver parameters ─────────────────────────────────────────────────────
TOL        = 0.01      # 1% optimality gap
TIME_LIMIT = 600       # 10 minutes per algorithm per instance
DATA_PATH  = '../dataset.xlsx'

print(f'Testing {len(CONFIGS)} configurations')
print(f'Tolerance: {TOL*100:.0f}%  |  Time limit: {TIME_LIMIT}s per solver')

## 2. Run both algorithms on each instance

For each configuration:
1. Build the instance and solve the nominal problem (used as warm-start for both algorithms).
2. Run C&CG and BDCP with the same warm-start and time limit.
3. Record key metrics for comparison.

In [ ]:
results = []

for (In, Jn, Rn, v_scale, w, Gamma, Hn, label) in CONFIGS:
    print(f'\n--- {label}: I={In} J={Jn} R={Rn} v={v_scale} w={w} Γ={Gamma} H={Hn} ---')

    # Build instance
    inst = instancemaker(In, Jn, Rn, v_scale, w, data_path=DATA_PATH)

    # Nominal solve (warm-start; Section 3, Algorithm 1 initialisation)
    nom = solve_nominal(inst)
    x_nom = nom['x_jr']
    print(f'  Nominal profit: {nom["profit"]:,.1f}  (runtime: {nom["runtime"]:.1f}s)')

    # ── C&CG ──────────────────────────────────────────────────────────────
    ccg = solve_CCG(
        inst, Gamma, Hn,
        x_init=x_nom,
        tol=TOL,
        time_limit=TIME_LIMIT,
        L_init=-abs(nom['profit']) * 2,  # valid lower bound
        verbose=False,
    )
    print(f'  CCG  | profit_LB={ccg["profit_LB"]:,.1f} | converged={ccg["converged"]} '
          f'| iters={ccg["n_iter"]} | blocks={ccg["n_blocks"]} | {ccg["runtime"]:.1f}s')

    # ── BDCP ──────────────────────────────────────────────────────────────
    bdcp = solve_BDCP(
        inst, Gamma, Hn,
        x_init=x_nom,
        tol=TOL,
        time_limit=TIME_LIMIT,
        verbose=False,
    )
    print(f'  BDCP | profit_LB={bdcp["profit_LB"]:,.1f} | converged={bdcp["converged"]} '
          f'| iters={bdcp["n_iter"]} | {bdcp["runtime"]:.1f}s')

    # ── Consistency metrics ───────────────────────────────────────────────
    abs_diff = abs(ccg['profit_LB'] - bdcp['profit_LB'])
    ref      = max(abs(ccg['profit_LB']), 1e-6)
    rel_diff = abs_diff / ref
    consistent = rel_diff <= TOL or abs_diff <= 5

    results.append({
        'label':           label,
        'I': In, 'J': Jn, 'R': Rn,
        'v_scale':         v_scale,
        'w':               w,
        'Gamma':           Gamma,
        'Hn':              Hn,
        'nominal_profit':  nom['profit'],
        'ccg_profit':      ccg['profit_LB'],
        'bdcp_profit':     bdcp['profit_LB'],
        'abs_diff':        abs_diff,
        'rel_diff_pct':    rel_diff * 100,
        'ccg_converged':   ccg['converged'],
        'bdcp_converged':  bdcp['converged'],
        'ccg_iters':       ccg['n_iter'],
        'bdcp_iters':      bdcp['n_iter'],
        'ccg_blocks':      ccg['n_blocks'],
        'ccg_runtime':     ccg['runtime'],
        'bdcp_runtime':    bdcp['runtime'],
        'consistent':      consistent,
    })

print('\nDone.')

## 3. Consistency report

In [ ]:
df = pd.DataFrame(results)

display_cols = [
    'label', 'Gamma', 'Hn',
    'nominal_profit', 'ccg_profit', 'bdcp_profit',
    'abs_diff', 'rel_diff_pct',
    'ccg_converged', 'bdcp_converged',
    'ccg_iters', 'bdcp_iters', 'ccg_blocks',
    'ccg_runtime', 'bdcp_runtime',
    'consistent',
]

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)

print('=== Consistency Results ===')
print(df[display_cols].to_string(index=False))

n_pass  = df['consistent'].sum()
n_total = len(df)
print(f'\nPASS: {n_pass}/{n_total} instances consistent within {TOL*100:.0f}% or ±5 units.')

if n_pass < n_total:
    print('\nFailing instances:')
    print(df.loc[~df['consistent'], ['label', 'ccg_profit', 'bdcp_profit', 'rel_diff_pct']])

## 4. First-stage solution comparison

Beyond objective values, we check whether the open facilities and capacity levels
chosen by CCG and BDCP are the same.  Different solutions can yield the same profit
(alternative optima), so we compare both the solutions and their recourse costs.

In [ ]:
from rcflp import worst_case_disruption, evaluate_second_stage

# Re-run on the first config in detail for illustration
In, Jn, Rn, v_scale, w, Gamma, Hn, label = CONFIGS[1]  # small-Γ2
inst = instancemaker(In, Jn, Rn, v_scale, w, data_path=DATA_PATH)
nom  = solve_nominal(inst)

ccg  = solve_CCG(inst, Gamma, Hn, x_init=nom['x_jr'], tol=TOL,
                 time_limit=TIME_LIMIT, L_init=-abs(nom['profit'])*2)
bdcp = solve_BDCP(inst, Gamma, Hn, x_init=nom['x_jr'], tol=TOL,
                  time_limit=TIME_LIMIT)

print(f'Instance: {label}  (I={In}, J={Jn}, R={Rn}, Γ={Gamma}, H={Hn})')
print()

# Open facilities and capacity levels
R = inst['R']
print('Open facilities (j, r) where x_jr ≈ 1:')
ccg_open  = [(j,r) for (j,r),v in ccg['x_jr'].items()  if v > 0.5]
bdcp_open = [(j,r) for (j,r),v in bdcp['x_jr'].items() if v > 0.5]
print(f'  CCG  : {sorted(ccg_open)}')
print(f'  BDCP : {sorted(bdcp_open)}')
print(f'  Same : {sorted(ccg_open) == sorted(bdcp_open)}')
print()

# Cross-evaluate: apply each other's x under worst-case scenario
eps_ccg,  rc_ccg  = worst_case_disruption(inst, ccg['x_jr'],  Gamma, Hn)
eps_bdcp, rc_bdcp = worst_case_disruption(inst, bdcp['x_jr'], Gamma, Hn)

fixed_ccg  = sum(inst['fixed_cost'][j,r] * ccg['x_jr'][j,r]  for j in inst['J'] for r in R)
fixed_bdcp = sum(inst['fixed_cost'][j,r] * bdcp['x_jr'][j,r] for j in inst['J'] for r in R)

print('Worst-case recourse cost (excl. fixed):')
print(f'  CCG  solution: {rc_ccg:,.2f}  (fixed={fixed_ccg:,.2f}, total={rc_ccg+fixed_ccg:,.2f})')
print(f'  BDCP solution: {rc_bdcp:,.2f}  (fixed={fixed_bdcp:,.2f}, total={rc_bdcp+fixed_bdcp:,.2f})')
print()
print(f'CCG  reported UB: {ccg["UB"]:,.2f}   profit_LB: {ccg["profit_LB"]:,.2f}')
print(f'BDCP reported UB: {bdcp["UB"]:,.2f}  profit_LB: {bdcp["profit_LB"]:,.2f}')

## 5. Summary

The table above reports:
- **`consistent`**: True if the relative difference in optimal profit is ≤ 1% or the absolute difference ≤ 5.
- **`rel_diff_pct`**: Percentage difference between CCG and BDCP profits.
- **`ccg_iters` / `bdcp_iters`**: Number of iterations to convergence.

Differences within the solver tolerance (1%) are expected and do not indicate a bug.
Larger discrepancies may signal a time-limit issue (algorithm did not converge) or a
code inconsistency between the two formulations.